# S&P 500 — Occupation-Level Wage Exposure Analysis (beta_new)

Estimates the total wage value at risk using the **Eloundou β (beta_new)** occupation-level
exposure score. beta_new is a weighted average of task-level β scores and ranges from 0 (no
exposure) to 1 (fully exposed).

**Pipeline**
1. Load company workforce data (`sp500_company_data.parquet`)
2. Load occupation-level Eloundou scores (`occ_level_new.csv`)
3. Aggregate workforce → occupation-level worker counts and wages
4. Merge with beta_new scores
5. Compute occupation wage exposure value = n_workers × avg_wage × beta_new
6. Run diagnostics
7. Aggregate across the S&P 500 and apply automation scenarios (25 %, 35 %, 45 %)
8. Save results

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

print("Libraries loaded ✓")

Libraries loaded ✓


## 1 · Configuration — File Paths

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = "/Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500"

PARQUET_PATH = os.path.join(BASE, "data",          "sp500_company_data.parquet")
OCC_PATH     = os.path.join(BASE, "Eloundou_New",  "occ_level_new.csv")
OUTPUT_PATH  = os.path.join(BASE, "Eloundou_New",  "occ_value_exposure_summary.csv")

for label, path in [("Company data",    PARQUET_PATH),
                    ("Occ-level scores", OCC_PATH)]:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {label:22s}  {path}")
    if not exists:
        raise FileNotFoundError(f"Required file not found: {path}")

  ✓  Company data            /Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500/data/sp500_company_data.parquet
  ✓  Occ-level scores        /Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500/Eloundou_New/occ_level_new.csv


## 2 · Load Data

In [3]:
# ── 2.1 Company workforce data ────────────────────────────────────────────────
print("Loading company workforce data …", flush=True)
df_raw = pd.read_parquet(PARQUET_PATH)

print(f"  Rows: {len(df_raw):,}")
print(f"  Columns: {df_raw.columns.tolist()}")

# ── Column auto-detection ─────────────────────────────────────────────────────
REQUIRED = {
    "occ_id"  : ["onet_code", "O*NET-SOC Code", "onet", "soc_code"],
    "wage"    : ["salary", "wage", "annual_wage", "mean_wage"],
    "weight"  : ["weight", "weights", "n_workers", "emp_count"],
}

resolved = {}
for field, candidates in REQUIRED.items():
    found = next((c for c in candidates if c in df_raw.columns), None)
    if found is None:
        raise KeyError(
            f"Cannot find column for '{field}'. "
            f"Expected one of {candidates}. "
            f"Available columns: {df_raw.columns.tolist()}"
        )
    resolved[field] = found
    print(f"  Detected '{field}' → '{found}'")

OCC_COL    = resolved["occ_id"]
WAGE_COL   = resolved["wage"]
WEIGHT_COL = resolved["weight"]

print("\nCompany data loaded ✓")

Loading company workforce data …
  Rows: 16,016,673
  Columns: ['user_id', 'position_id', 'rcid', 'seniority', 'country', 'salary', 'onet_code', 'startdate', 'enddate', 'weight', 'highest_degree', 'sex_predicted', 'ethnicity_predicted', 'ticker', 'naics_code', 'exchange_name', 'company']
  Detected 'occ_id' → 'onet_code'
  Detected 'wage' → 'salary'
  Detected 'weight' → 'weight'

Company data loaded ✓


In [4]:
# ── 2.2 Occupation-level Eloundou scores (beta_new) ──────────────────────────
print("Loading occupation-level scores …", flush=True)
df_occ_scores = pd.read_csv(OCC_PATH)

# Auto-detect occupation ID column
OCC_SCORE_CANDIDATES = ["O*NET-SOC Code", "onet_soc_code", "onet_code", "soc_code"]
occ_score_occ_col = next((c for c in OCC_SCORE_CANDIDATES if c in df_occ_scores.columns), None)
if occ_score_occ_col is None:
    raise KeyError(f"Cannot find occupation ID in occ_level_new. Got: {df_occ_scores.columns.tolist()}")

# Require beta_new explicitly — fail loudly if absent
if "beta_new" not in df_occ_scores.columns:
    raise RuntimeError(
        "FATAL: 'beta_new' column not found in occ_level_new.csv. "
        "Run the 0.5 notebook first to generate it."
    )

# Auto-detect title column if present
title_col = next((c for c in ["Title", "title", "occ_title"] if c in df_occ_scores.columns), None)

keep_cols = [occ_score_occ_col, "beta_new"]
if title_col:
    keep_cols.insert(1, title_col)

df_occ_scores = (
    df_occ_scores[keep_cols]
    .rename(columns={occ_score_occ_col: "occ_id"})
    .copy()
)
df_occ_scores["occ_id"] = df_occ_scores["occ_id"].astype(str).str.strip()

n_with_beta = df_occ_scores["beta_new"].notna().sum()
n_total_occ = len(df_occ_scores)

print(f"  Detected occ ID column : '{occ_score_occ_col}'")
print(f"  Title column           : '{title_col}'")
print(f"  Occupations total      : {n_total_occ:,}")
print(f"  Occupations with beta_new : {n_with_beta:,}  ({100*n_with_beta/n_total_occ:.1f}%)")
print(f"\n  beta_new distribution:")
print(df_occ_scores["beta_new"].describe().to_string())
print("\nOccupation-level scores loaded ✓")

Loading occupation-level scores …
  Detected occ ID column : 'O*NET-SOC Code'
  Title column           : 'Title'
  Occupations total      : 923
  Occupations with beta_new : 923  (100.0%)

  beta_new distribution:
count   923.0000
mean      0.3344
std       0.2425
min       0.0000
25%       0.1111
50%       0.3158
75%       0.5189
max       1.0000

Occupation-level scores loaded ✓


In [5]:
# ── 2.3 Quick preview ────────────────────────────────────────────────────────
pd.set_option("display.float_format", "{:.4f}".format)
print("Sample of occ_level_new (first 10 rows with beta_new):")
print(df_occ_scores.dropna(subset=["beta_new"]).head(10).to_string(index=False))

Sample of occ_level_new (first 10 rows with beta_new):
    occ_id                               Title  beta_new
11-1011.00                    Chief Executives    0.4302
11-1011.03       Chief Sustainability Officers    0.8056
11-1021.00     General and Operations Managers    0.5600
11-1031.00                         Legislators    0.3667
11-2011.00 Advertising and Promotions Managers    0.6053
11-2021.00                  Marketing Managers    0.6786
11-2022.00                      Sales Managers    0.5952
11-2032.00           Public Relations Managers    0.7500
11-2033.00                Fundraising Managers    0.7812
11-3012.00    Administrative Services Managers    0.5714


## 3 · Aggregate Workforce → Occupation-Level Worker Counts and Wages

In [6]:
# ── 3.1  Drop rows missing the occupation identifier ─────────────────────────
before = len(df_raw)
df_work = df_raw.dropna(subset=[OCC_COL]).copy()
after  = len(df_work)
print(f"Dropped {before - after:,} rows with missing occupation code "
      f"({(before - after) / before:.1%} of total)")

# ── 3.2  Impute missing wages from occupation-weighted mean ───────────────────
df_work[WEIGHT_COL] = df_work[WEIGHT_COL].fillna(1.0)

occ_mean_wage = (
    df_work.dropna(subset=[WAGE_COL])
    .groupby(OCC_COL)
    .apply(lambda g: np.average(g[WAGE_COL], weights=g[WEIGHT_COL]))
    .rename("mean_occ_wage")
    .reset_index()
)
occ_mean_wage.columns = [OCC_COL, "mean_occ_wage"]

df_work = df_work.merge(occ_mean_wage, on=OCC_COL, how="left")
df_work[WAGE_COL] = df_work[WAGE_COL].fillna(df_work["mean_occ_wage"])

n_still_missing = df_work[WAGE_COL].isna().sum()
print(f"After imputation: {n_still_missing:,} rows still missing wage "
      f"(excluded from wage mass).")

# ── 3.3  Occupation-level aggregation ────────────────────────────────────────
def wtd_mean(g):
    mask = g[WAGE_COL].notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(g.loc[mask, WAGE_COL], weights=g.loc[mask, WEIGHT_COL])

occ_agg = (
    df_work.groupby(OCC_COL)
    .apply(lambda g: pd.Series({
        "n_workers": g[WEIGHT_COL].sum(),
        "avg_wage" : wtd_mean(g),
    }))
    .reset_index()
)
occ_agg.columns = ["occ_id", "n_workers", "avg_wage"]
occ_agg["occ_id"] = occ_agg["occ_id"].astype(str).str.strip()

before_occ = len(occ_agg)
occ_agg = occ_agg.dropna(subset=["avg_wage"])
after_occ = len(occ_agg)
print(f"\nOccupations after aggregation : {before_occ:,}  "
      f"(dropped {before_occ - after_occ:,} with no wage data)")

total_wage_mass = (occ_agg["n_workers"] * occ_agg["avg_wage"]).sum()
print(f"\nTotal workforce wage mass  : ${total_wage_mass:,.0f}")
print(f"Occupations covered        : {len(occ_agg):,}")
print(f"Weighted workers covered   : {occ_agg['n_workers'].sum():,.0f}")
print("\nOccupation aggregation ✓")

Dropped 6,712 rows with missing occupation code (0.0% of total)
After imputation: 0 rows still missing wage (excluded from wage mass).

Occupations after aggregation : 1,009  (dropped 0 with no wage data)

Total workforce wage mass  : $1,303,402,913,650
Occupations covered        : 1,009
Weighted workers covered   : 18,649,010

Occupation aggregation ✓


## 4 · Merge Workforce with beta_new Scores

In [7]:
# ── Merge occupation aggregates with Eloundou beta_new ───────────────────────
df_merged = occ_agg.merge(df_occ_scores, on="occ_id", how="left")

n_occ_total    = len(df_merged)
n_occ_matched  = df_merged["beta_new"].notna().sum()
n_occ_missing  = n_occ_total - n_occ_matched

wage_mass_with_beta    = (df_merged.dropna(subset=["beta_new"])
                          .eval("wm = n_workers * avg_wage")["wm"].sum())
wage_mass_without_beta = (df_merged[df_merged["beta_new"].isna()]
                          .eval("wm = n_workers * avg_wage")["wm"].sum())

print(f"Occupations in workforce   : {n_occ_total:,}")
print(f"Matched to beta_new        : {n_occ_matched:,}  ({100*n_occ_matched/n_occ_total:.1f}%)")
print(f"No beta_new score          : {n_occ_missing:,}  ({100*n_occ_missing/n_occ_total:.1f}%)")
print()
print(f"Wage mass WITH beta_new    : ${wage_mass_with_beta:,.0f}  "
      f"({100*wage_mass_with_beta/total_wage_mass:.1f}% of total)")
print(f"Wage mass WITHOUT beta_new : ${wage_mass_without_beta:,.0f}  "
      f"({100*wage_mass_without_beta/total_wage_mass:.1f}% of total)")
print("\nMerge complete ✓")

Occupations in workforce   : 1,009
Matched to beta_new        : 916  (90.8%)
No beta_new score          : 93  (9.2%)

Wage mass WITH beta_new    : $1,192,750,938,159  (91.5% of total)
Wage mass WITHOUT beta_new : $110,651,975,492  (8.5% of total)

Merge complete ✓


## 5 · Diagnostics — beta_new Distribution Across the S&P 500 Workforce

In [8]:
# ── Only work with occupations that have a beta_new score ────────────────────
df_scored = df_merged.dropna(subset=["beta_new"]).copy()
df_scored["wage_mass"] = df_scored["n_workers"] * df_scored["avg_wage"]

print("=== beta_new distribution (occupation-level, unweighted) ===")
print(df_scored["beta_new"].describe().to_string())

print("\n=== beta_new distribution (worker-weighted) ===")
wtd_mean_beta = np.average(df_scored["beta_new"], weights=df_scored["n_workers"])
wtd_std_beta  = np.sqrt(np.average(
    (df_scored["beta_new"] - wtd_mean_beta) ** 2,
    weights=df_scored["n_workers"]
))
print(f"  Worker-weighted mean beta_new : {wtd_mean_beta:.4f}")
print(f"  Worker-weighted std  beta_new : {wtd_std_beta:.4f}")

print("\n=== beta_new distribution (wage-mass-weighted) ===")
wm_mean_beta = np.average(df_scored["beta_new"], weights=df_scored["wage_mass"])
print(f"  Wage-mass-weighted mean beta_new : {wm_mean_beta:.4f}")

# ── beta_new quintiles ────────────────────────────────────────────────────────
bins   = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.01]
labels = ["0-10%","10-20%","20-30%","30-40%","40-50%",
          "50-60%","60-70%","70-80%","80-90%","90-100%"]
df_scored["beta_bin"] = pd.cut(df_scored["beta_new"], bins=bins, labels=labels, right=False)

bin_stats = df_scored.groupby("beta_bin", observed=False).agg(
    n_occs    = ("occ_id",    "count"),
    n_workers = ("n_workers", "sum"),
    wage_mass = ("wage_mass", "sum"),
).reset_index()
bin_stats["wm_share"] = bin_stats["wage_mass"] / bin_stats["wage_mass"].sum()

print("\n=== Occupations by beta_new decile ===")
print(f"{'Decile':<12} {'N occs':>8} {'Workers':>12} {'Wage mass ($)':>18} {'WM share':>10}")
print("-" * 65)
for _, r in bin_stats.iterrows():
    print(f"{str(r['beta_bin']):<12} {r['n_occs']:>8,} {r['n_workers']:>12,.0f} "
          f"${r['wage_mass']:>17,.0f} {r['wm_share']:>9.1%}")
print("\nDiagnostics ✓")

=== beta_new distribution (occupation-level, unweighted) ===
count   916.0000
mean      0.3359
std       0.2426
min       0.0000
25%       0.1125
50%       0.3176
75%       0.5208
max       1.0000

=== beta_new distribution (worker-weighted) ===
  Worker-weighted mean beta_new : 0.5523
  Worker-weighted std  beta_new : 0.2192

=== beta_new distribution (wage-mass-weighted) ===
  Wage-mass-weighted mean beta_new : 0.5930

=== Occupations by beta_new decile ===
Decile         N occs      Workers      Wage mass ($)   WM share
-----------------------------------------------------------------
0-10%             212    1,071,797 $   33,418,961,386      2.8%
10-20%            110      588,715 $   25,833,135,811      2.2%
20-30%            121      550,002 $   27,893,360,950      2.3%
30-40%             99    1,577,513 $   80,237,481,405      6.7%
40-50%            111    1,294,271 $  110,884,142,913      9.3%
50-60%            104    3,819,878 $  261,127,358,432     21.9%
60-70%             84

## 6 · Compute Occupation-Level Exposure Value

In [9]:
# ── exposure_value = n_workers × avg_wage × beta_new ─────────────────────────
# beta_new is a continuous score in [0, 1] representing the occupation's
# share of AI-exposed work. exposure_value is the dollar value of wage mass
# that lies in tasks classified as E1/E2 (weighted by task-time shares).

df_scored["exposure_value"] = df_scored["wage_mass"] * df_scored["beta_new"]

total_wage_mass_scored = df_scored["wage_mass"].sum()
total_exposure_value   = df_scored["exposure_value"].sum()
non_exposure_value     = total_wage_mass_scored - total_exposure_value

print("=== Occupation-level wage exposure (beta_new) ===")
print(f"  Total wage mass (scored occs)       : ${total_wage_mass_scored:,.0f}")
print(f"  Total exposure value (×beta_new)    : ${total_exposure_value:,.0f}  "
      f"({100*total_exposure_value/total_wage_mass_scored:.1f}%)")
print(f"  Non-exposed wage value              : ${non_exposure_value:,.0f}  "
      f"({100*non_exposure_value/total_wage_mass_scored:.1f}%)")
print()

# Also show coverage relative to the full workforce wage mass
print(f"  Scored occs wage mass / all occs wage mass : "
      f"{100*total_wage_mass_scored/total_wage_mass:.1f}%")
print("\nExposure values computed ✓")

=== Occupation-level wage exposure (beta_new) ===
  Total wage mass (scored occs)       : $1,192,750,938,159
  Total exposure value (×beta_new)    : $707,242,531,414  (59.3%)
  Non-exposed wage value              : $485,508,406,745  (40.7%)

  Scored occs wage mass / all occs wage mass : 91.5%

Exposure values computed ✓


## 7 · Automation Scenarios (25 %, 35 %, 45 %)

In [10]:
# Interpretation:
#   At automation rate X, the wage value automated per occupation =
#     n_workers × avg_wage × beta_new × X
#   Summed across all occupations this gives the total automated wage value.

SCENARIOS = {"25%": 0.25, "35%": 0.35, "45%": 0.45}

rows = []
for label, rate in SCENARIOS.items():
    auto_value = total_exposure_value * rate
    rows.append({
        "Scenario"              : f"Scenario {label}",
        "Automation rate"       : rate,
        "Automated wage ($)"    : auto_value,
        "Share of scored WM (%)": 100 * auto_value / total_wage_mass_scored,
        "Share of total WM (%)" : 100 * auto_value / total_wage_mass,
    })

df_scenarios = pd.DataFrame(rows)

print("=== Automation scenario results ===\n")
for _, r in df_scenarios.iterrows():
    print(f"  {r['Scenario']}  ({r['Automation rate']:.0%} of exposure value)")
    print(f"    Automated wage value     : ${r['Automated wage ($)']:,.0f}")
    print(f"    Share of scored WM       : {r['Share of scored WM (%)']:.1f}%")
    print(f"    Share of total WM        : {r['Share of total WM (%)']:.1f}%")
    print()

print("Automation scenarios ✓")

=== Automation scenario results ===

  Scenario 25%  (25% of exposure value)
    Automated wage value     : $176,810,632,853
    Share of scored WM       : 14.8%
    Share of total WM        : 13.6%

  Scenario 35%  (35% of exposure value)
    Automated wage value     : $247,534,885,995
    Share of scored WM       : 20.8%
    Share of total WM        : 19.0%

  Scenario 45%  (45% of exposure value)
    Automated wage value     : $318,259,139,136
    Share of scored WM       : 26.7%
    Share of total WM        : 24.4%

Automation scenarios ✓


## 8 · Summary Table

In [11]:
print("=" * 90)
print("  OCCUPATION-LEVEL WAGE EXPOSURE SUMMARY  (S&P 500 workforce, beta_new)")
print("=" * 90)
print(f"\n  Total workforce wage mass                : ${total_wage_mass:,.0f}")
print(f"  Wage mass in scored occupations          : ${total_wage_mass_scored:,.0f}   "
      f"({100*total_wage_mass_scored/total_wage_mass:.1f}% of total)")
print(f"\n  Exposure value  (WM × beta_new)          : ${total_exposure_value:,.0f}   "
      f"({100*total_exposure_value/total_wage_mass_scored:.1f}% of scored WM)")
print(f"  Non-exposed value                        : ${non_exposure_value:,.0f}   "
      f"({100*non_exposure_value/total_wage_mass_scored:.1f}% of scored WM)")
print()
print(f"  Worker-weighted mean beta_new            : {wtd_mean_beta:.4f}")
print(f"  Wage-mass-weighted mean beta_new         : {wm_mean_beta:.4f}")
print()
print("  Automated wage value by scenario:")
print(f"  {'Scenario':<16} {'Automated ($)':>18} {'% of scored WM':>16} {'% of total WM':>15}")
print("  " + "-" * 68)
for _, r in df_scenarios.iterrows():
    print(f"  {r['Scenario']:<16} ${r['Automated wage ($)']:>17,.0f} "
          f"{r['Share of scored WM (%)']:>15.1f}% "
          f"{r['Share of total WM (%)']:>14.1f}%")
print("=" * 90)
print("\nSummary ✓")

  OCCUPATION-LEVEL WAGE EXPOSURE SUMMARY  (S&P 500 workforce, beta_new)

  Total workforce wage mass                : $1,303,402,913,650
  Wage mass in scored occupations          : $1,192,750,938,159   (91.5% of total)

  Exposure value  (WM × beta_new)          : $707,242,531,414   (59.3% of scored WM)
  Non-exposed value                        : $485,508,406,745   (40.7% of scored WM)

  Worker-weighted mean beta_new            : 0.5523
  Wage-mass-weighted mean beta_new         : 0.5930

  Automated wage value by scenario:
  Scenario              Automated ($)   % of scored WM   % of total WM
  --------------------------------------------------------------------
  Scenario 25%     $  176,810,632,853            14.8%           13.6%
  Scenario 35%     $  247,534,885,995            20.8%           19.0%
  Scenario 45%     $  318,259,139,136            26.7%           24.4%

Summary ✓


## 9 · Top Occupations by Exposure Value and by beta_new Score

In [12]:
TOP_N = 20

title_display = "Title" if "Title" in df_scored.columns else "occ_id"

display_cols = ["occ_id", title_display, "n_workers", "avg_wage",
                "wage_mass", "beta_new", "exposure_value"]
# deduplicate if title_display == "occ_id"
display_cols = list(dict.fromkeys(display_cols))

# ── Top 20 by exposure_value (absolute dollar exposure) ──────────────────────
top_ev = (
    df_scored[display_cols]
    .sort_values("exposure_value", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)
top_ev.index += 1
top_ev["exp_share (%)"] = top_ev["exposure_value"] / total_exposure_value * 100

print(f"\n{'='*90}")
print(f"  Top {TOP_N} occupations by EXPOSURE VALUE  (n_workers × avg_wage × beta_new)")
print(f"{'='*90}")
print(top_ev.to_string(float_format="${:,.0f}".format))

# ── Top 20 by beta_new score ──────────────────────────────────────────────────
top_beta = (
    df_scored[display_cols]
    .sort_values("beta_new", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)
top_beta.index += 1

print(f"\n{'='*90}")
print(f"  Top {TOP_N} occupations by BETA_NEW SCORE")
print(f"{'='*90}")
print(top_beta.to_string(float_format="{:,.4f}".format))

# ── Bottom 20 by beta_new (lowest exposure) ───────────────────────────────────
bot_beta = (
    df_scored[display_cols]
    .sort_values("beta_new", ascending=True)
    .head(TOP_N)
    .reset_index(drop=True)
)
bot_beta.index += 1

print(f"\n{'='*90}")
print(f"  Bottom {TOP_N} occupations by BETA_NEW SCORE  (least exposed)")
print(f"{'='*90}")
print(bot_beta.to_string(float_format="{:,.4f}".format))


  Top 20 occupations by EXPOSURE VALUE  (n_workers × avg_wage × beta_new)
        occ_id                                                                                             Title  n_workers  avg_wage        wage_mass  beta_new  exposure_value  exp_share (%)
1   15-1252.00                                                                               Software Developers $1,263,397   $82,873 $104,701,871,250        $1 $74,393,434,835            $11
2   15-1299.09                                                           Information Technology Project Managers   $566,431   $97,391  $55,165,287,525        $1 $38,090,317,577             $5
3   15-1299.08                                                             Computer Systems Engineers/Architects   $446,691   $87,942  $39,283,007,533        $1 $30,163,737,927             $4
4   11-2021.00                                                                                Marketing Managers   $360,064  $110,201  $39,679,370,007       

## 10 · Save Results